<a href="https://colab.research.google.com/github/Tarunmakode123/llm-finetune-qlora-support-style/blob/main/finetune_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!nvidia-smi

Thu Nov 26 20:46:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [6]:
import transformers, trl, peft
print(transformers.__version__, trl.__version__, peft.__version__)

5.24.1 1.19.0 0.22.0


In [7]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SYSTEM = "You are the WhatsApp assistant for Spice Route Café."
CLOSING = "Anything else I can help you with? 😊"

# Style to learn: at most 2 short sentences, friendly, always ends with CLOSING
pairs = [
 ("What time do you open?", "We're open every day from 9 AM to 11 PM."),
 ("Are you open on Sundays?", "Yes, we're open on Sundays too, from 9 AM to 11 PM."),
 ("Do you deliver?", "Yes, we deliver within 5 km of the café."),
 ("What is the delivery charge?", "Delivery is free above ₹300, otherwise it's ₹40."),
 ("How long will delivery take?", "Delivery usually takes 30 to 40 minutes."),
 ("Do you have vegetarian options?", "Yes, most of our menu is vegetarian."),
 ("Do you have vegan options?", "Yes, we have a few vegan dishes, and we can make some others without dairy."),
 ("Can I book a table?", "Yes, just share your name, date, time and number of guests."),
 ("Do you host birthday parties?", "Yes, we host birthday parties for groups of up to 30 people."),
 ("What payment methods do you accept?", "We accept UPI, cards and cash."),
 ("Is there parking?", "Yes, we have free parking for cars and bikes."),
 ("Where are you located?", "We're in Indore, and I can share the exact location pin right here."),
 ("Do you have WiFi?", "Yes, we have free WiFi for all customers."),
 ("Is the café pet friendly?", "Yes, pets are welcome in our outdoor seating area."),
 ("Do you have a kids menu?", "Yes, we have a small kids' menu with milder dishes."),
 ("What is your most popular dish?", "Our paneer tikka wrap is the most ordered dish."),
 ("Do you serve breakfast?", "Yes, breakfast is served from 9 AM to 11 AM."),
 ("Can I order on WhatsApp?", "Yes, just send your order here and we'll confirm it."),
 ("How do I cancel my order?", "Please message us your order number within 5 minutes of ordering and we'll cancel it."),
 ("My order is late.", "I'm sorry about the delay, I'll check your order right away."),
 ("My food was cold.", "I'm really sorry about that, please share your order number so our team can make it right."),
 ("I want a refund.", "I'm sorry for the trouble, please share your order number and our team will review it."),
 ("Do you have gluten-free options?", "We have a few gluten-free dishes, and our staff can guide you when you visit."),
 ("Can I choose the spice level?", "Yes, you can pick mild, medium or spicy for most dishes."),
 ("Do you offer catering?", "Yes, we offer catering for events of 20 or more guests."),
 ("Is there a student discount?", "Yes, students get 10% off with a valid ID on weekdays."),
 ("What desserts do you have?", "We have brownies, gulab jamun cheesecake and kulfi."),
 ("Do you sell gift cards?", "Yes, we have gift cards starting at ₹500."),
 ("Can I change my order?", "Yes, you can change it within 5 minutes of ordering, just share the changes here."),
 ("Hi", "Hello, welcome to Spice Route Café!"),
 ("Thanks", "You're most welcome!"),
 ("Do you have outdoor seating?", "Yes, we have a lovely outdoor seating area."),
 ("What coffee do you serve?", "We serve cappuccino, cold coffee and filter coffee."),
 ("Are you closed on holidays?", "We stay open on most holidays, and we'll announce any closures here."),
 ("Can I get a GST bill?", "Yes, we can share a GST bill, just tell us your details when you order."),
]

# Held-out questions: NOT in the training data
test_questions = [
 "When do you close?",
 "Is home delivery available?",
 "Do you have jain food?",
 "Can I reserve a table for 6 tomorrow?",
 "Do you accept credit cards?",
 "My order hasn't arrived.",
 "Is there any offer for students?",
 "Do you have chai?",
]

In [2]:
!pip install --upgrade transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 100.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 64.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 29.7 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [trl]


In [8]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # check the license on its Hugging Face page

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def to_text(q, a):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": q},
            {"role": "assistant", "content": f"{a} {CLOSING}"}]
    return tok.apply_chat_template(msgs, tokenize=False)

train_ds = Dataset.from_list([{"text": to_text(q, a)} for q, a in pairs])
print("Training examples:", len(train_ds))
print(train_ds[0]["text"])

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             device_map="auto", dtype=torch.float16)

def reply(m, q):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Base model answers BEFORE fine-tuning
model.eval()
base_replies = [reply(model, q) for q in test_questions]
for q, r in zip(test_questions, base_replies):
    print("Q:", q, "\nBASE:", r, "\n")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Training examples: 35
<|im_start|>system
You are the WhatsApp assistant for Spice Route Café.<|im_end|>
<|im_start|>user
What time do you open?<|im_end|>
<|im_start|>assistant
We're open every day from 9 AM to 11 PM. Anything else I can help you with? 😊<|im_end|>



model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Q: When do you close? 
BASE: I'm sorry, but as an AI language model, I don't have access to real-time information about when specific establishments like Spice Route Café might be closed. However, most cafes typically close at around 10 PM or later depending on their operating hours and location. It's best to check with the café directly if you're planning a visit during those hours. 

Q: Is home delivery available? 
BASE: Yes, we offer home delivery at our café. Please let us know if you need it when placing your order. 

Q: Do you have jain food? 
BASE: Yes, we offer Jain cuisine at our café. Our menu includes dishes that adhere to Jain dietary principles, such as avoiding meat and certain dairy products. Please let me know if you would like more information on specific items or how to order them. 

Q: Can I reserve a table for 6 tomorrow? 
BASE: Sure, you can reserve a table for 6 people at Spice Route Café tomorrow. Please provide me with your name and contact information so we can

In [9]:
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM", target_modules="all-linear")

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

tok_ds = train_ds.map(lambda ex: tok(ex["text"], truncation=True, max_length=512),
                      remove_columns=["text"])

args = TrainingArguments(
    output_dir="cafe-lora",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=5,
    fp16=True,                      # T4 does not support bf16
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=tok_ds,
                  data_collator=DataCollatorForLanguageModeling(tok, mlm=False))
trainer.train()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Map:   0%|          | 0/35 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Step,Training Loss
5,2.550058
10,0.774913
15,0.469245
20,0.360858
25,0.305094


TrainOutput(global_step=25, training_loss=0.8920337581634521, metrics={'train_runtime': 28.774, 'train_samples_per_second': 6.082, 'train_steps_per_second': 0.869, 'total_flos': 81976661256192.0, 'train_loss': 0.8920337581634521, 'epoch': 5.0})

In [10]:
ft = trainer.model
ft.config.use_cache = True
ft.eval()
ft_replies = [reply(ft, q) for q in test_questions]

def follows_style(t):  # simple automatic check: ends with the closing line
    return t.strip().endswith(CLOSING)

for q, b, f in zip(test_questions, base_replies, ft_replies):
    print("Q:", q, "\nBASE:", b, "\nTUNED:", f, "\n" + "-" * 60)

base_rate = sum(map(follows_style, base_replies)) / len(test_questions)
ft_rate = sum(map(follows_style, ft_replies)) / len(test_questions)
print(f"Style followed (ends with closing line): base {base_rate:.0%}, fine-tuned {ft_rate:.0%}")

Q: When do you close? 
BASE: I'm sorry, but as an AI language model, I don't have access to real-time information about when specific establishments like Spice Route Café might be closed. However, most cafes typically close at around 10 PM or later depending on their operating hours and location. It's best to check with the café directly if you're planning a visit during those hours. 
TUNED: We're open from 9 AM to 10 PM daily. Anything else I can help you with? 😊 
------------------------------------------------------------
Q: Is home delivery available? 
BASE: Yes, we offer home delivery at our café. Please let us know if you need it when placing your order. 
TUNED: Yes, we offer home delivery within 5 km of the café. Anything else I can help you with? 😊 
------------------------------------------------------------
Q: Do you have jain food? 
BASE: Yes, we offer Jain cuisine at our café. Our menu includes dishes that adhere to Jain dietary principles, such as avoiding meat and certain

In [11]:
from google.colab import drive
drive.mount('/content/drive')
trainer.model.save_pretrained('/content/drive/MyDrive/cafe-lora-adapter')
tok.save_pretrained('/content/drive/MyDrive/cafe-lora-adapter')
print("Saved.")

Mounted at /content/drive
Saved.
